# Part 4: 딥러닝 모델에 양자화 적용하기


## 1. 환경 설정 및 라이브러리 설치(추천 런타임 유형:T4)


In [ ]:
# 라이브러리 설치 (버전 호환성 확보)
# !pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnx onnxruntime onnxscript numpy==1.26.4

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from copy import deepcopy
import time
import os
import numpy as np
import warnings

# 불필요한 경고 메시지 숨기기 (양자화 관련 경고가 많이 뜰 수 있음)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PyTorch Version: 2.9.0+cu126
Device: cuda


- Step 0 (Baseline): 아무것도 건드리지 않은 원본 모델을 만들고 성적표 확인 (비교 대상).
- Step 1 (Dynamic): 가장 쉬운 방법으로 Linear 레이어만 툭 던져서 양자화해보기.
- Step 2 (Static): (지금 보고 계신 코드) QuantStub 등을 써서 모델 구조를 바꾸고, 실제 데이터로 범위를 관찰해서 정밀하게 바꾸는 방법.
- Step 3 (ONNX): 배포용 포맷으로 바꿔서 외부 엔진에서 돌리는 법.

## 2. 데이터셋 준비 및 기본 모델 훈련 (FP32)

In [ ]:
# ======================================================================
# 🚀 0단계: 양자화 실습의 '기준선(Baseline)' 마련하기 — 왜 이 단계가 필수인가?
# ----------------------------------------------------------------------
# [핵심 목적 3가지]
# 1️⃣ "다이어트 전 몸무게 측정" → size_fp32
#    - 양자화는 모델을 '압축'하는 과정입니다. 
#    - 압축 전/후 용량 비교를 위해 반드시 원본 크기를 정확히 측정해야 합니다.
#    - "4배 줄었다"는 주장은 이 기준치에서 시작됩니다.
#
# 2️⃣ "건강 검진 결과표" → acc_fp32
#    - 양자화 후 정확도가 98%에서 95%로 떨어졌다면, 
#      "3%p 하락"이라는 의미 있는 결론을 낼 수 있습니다.
#    - 기준선 없이는 "95%가 좋은 건가 나쁜 건가?" 판단 불가.
#
# 3️⃣ "양자화 눈금자 준비" → Calibration 데이터
#    - 정적 양자화(Static Quantization) 시 각 레이어의 활성화 범위(min/max)를 
#      미리 측정해야 합니다. 이때 사용할 실제 데이터 샘플을 준비하는 단계입니다.
#    - 마치 체중계를 사용하기 전 "0점 조정"을 하는 것과 동일한 원리입니다.
# ======================================================================

# 1. 데이터셋 로드 (MNIST) — 양자화의 '현실적 테스트베드'
# ----------------------------------------------------------------------
# [왜 MNIST인가?]
# - 간단하지만 충분한 복잡도: 양자화 오류가 명확히 드러나는 최소한의 복잡도
# - 빠른 실험 반복 가능: 10분 내 완료 → 양자화 파라미터 튜닝에 최적
# - 표준 벤치마크: 다른 연구 결과와의 정확도 비교가 용이
#
# [양자화 관점에서의 특별한 의미: Calibration 데이터]
# 🔑 "정적 양자화"에서는 반드시 실제 데이터를 한 번 흘려보내며 
#    각 레이어의 입력/출력 분포를 관찰해야 합니다.
#    → 이 과정을 'Calibration'이라 하며, 
#    → Calibration에 사용되는 데이터를 'Calibration Dataset'이라 부릅니다.
#    → 여기서 로드한 train_dataset의 일부가 나중에 이 역할을 수행합니다.
#
#
# [양자화의 핵심 비결: 정규화(Normalization)]
# [데이터 전처리의 숨은 의미]
# - transforms.Normalize((0.1307,), (0.3081,))
#   → MNIST 전체 데이터셋의 평균/표준편차로 정규화
#   → 양자화 시 0 근처에 데이터가 몰리면 표현 범위 낭비 → 정규화로 분포 최적화
#   → "양자화 눈금자(Scale)를 더 효율적으로 설정하기 위한 사전 작업"
#
# 1. 왜 하는가?: 8비트 양자화는 데이터를 딱 256칸(0~255)짜리 방에 가두는 작업입니다.
# 2. 정규화를 안 하면?: 
#    - 데이터가 0 근처에만 다닥다닥 붙어 있어서 256칸 중 앞쪽 몇 칸만 쓰고 나머지는 텅 비게 됩니다.
#    - 미세한 소수점 차이들이 같은 칸(0번 칸)에 담겨 정보가 뭉개지는 '해상도 낭비'가 발생합니다.
# 3. 정규화를 하면?:
#    - 0에 뭉쳐있던 데이터들을 좌우로 쫙 펴서 256칸 전체를 골고루 사용하게 만듭니다.
#    - 결과적으로 '양자화 눈금자(Scale)'를 훨씬 정밀하게 설정할 수 있어 정확도 손실을 최소화합니다.
# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),          # PIL 이미지 → [0,1] 범위의 FloatTensor
    transforms.Normalize((0.1307,), (0.3081,))  # 전체 MNIST의 μ=0.1307, σ=0.3081로 정규화
])

# 훈련 데이터: 나중에 Calibration 용도로도 사용됨 (약 100~1000개 샘플만 사용)
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# 테스트 데이터: 양자화 전/후 성능 비교를 위한 고정된 평가 세트
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 배치 크기 64: 메모리 효율성과 훈련 안정성의 균형점
# (양자화 시에도 동일한 배치 크기 사용 → 공정한 비교를 위해)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# 2. 원본 모델 정의 (FP32: 32비트 부동소수점) — 양자화의 '황금 기준(Gold Standard)'
# ----------------------------------------------------------------------
# [왜 이 구조인가? 양자화 실습에 최적화된 설계]
# ✅ Conv → ReLU → Pooling 반복: 
#    - 실제 딥러닝 모델의 기본 블록 구조 반영
#    - 양자화 시 가장 민감한 레이어 조합 (Conv + ReLU)
# ✅ Fully Connected 레이어 포함:
#    - 행렬 곱셈의 양자화 오류를 관찰 가능
# ✅ 깊지 않은 구조 (2 Conv + 2 FC):
#    - 양자화 오류 누적이 적어, 단일 레이어의 영향 분석 용이
#
# [양자화 관점에서의 주의점]
# ⚠️ ReLU 위치: Conv 다음에 배치 → 
#    - 활성화 값이 항상 ≥0 → 양자화 범위를 [0, max]로 제한 가능 
#    - (부호 있는 8비트 대신 부호 없는 8비트 사용 시 메모리 2배 절약)
# ⚠️ Padding=1 사용: 
#    - 특징맵 크기 유지 → 양자화 시 공간 정보 손실 최소화
# ----------------------------------------------------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 입력: [1, 28, 28] → 출력: [32, 28, 28] (padding=1로 크기 유지)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        # 2x2 MaxPooling → 크기 1/2로 축소: [32, 28, 28] → [32, 14, 14]
        self.pool = nn.MaxPool2d(2, 2)
        # 입력: [32, 14, 14] → 출력: [64, 14, 14]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Flatten: [64, 7, 7] → 64*7*7 = 3136차원 벡터
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        # 최종 출력: 10개 클래스 (0~9 숫자)
        self.fc2 = nn.Linear(128, 10)
        # ReLU: 음수를 0으로 클리핑 → 양자화 범위를 [0, ∞)로 제한 (중요!)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Layer 1: Conv → ReLU → Pooling
        x = self.pool(self.relu(self.conv1(x)))  # [1,28,28] → [32,14,14]
        # Layer 2: Conv → ReLU → Pooling
        x = self.pool(self.relu(self.conv2(x)))  # [32,14,14] → [64,7,7]
        # Flatten for FC layers
        x = x.reshape(-1, 64 * 7 * 7)            # [64,7,7] → [3136]
        # FC Layer 1 + ReLU
        x = self.relu(self.fc1(x))               # [3136] → [128]
        # FC Layer 2 (출력 레이어, Softmax는 손실 함수 내부에서 처리)
        x = self.fc2(x)                          # [128] → [10]
        return x


# 3. 모델 훈련 함수 — "양자화 전 반드시 해야 할 일: 충분한 학습"
# ----------------------------------------------------------------------
# [왜 훈련이 필수인가?]
# ❌ 실패 사례: 
#    - 원본 모델 정확도 85% → 양자화 후 83% → "2%p 하락"이라며 양자화 탓
#    - 실제 원인: 원본 모델 자체가 제대로 학습되지 않음 (과소적합)
# ✅ 성공 조건:
#    - 원본 모델 정확도 ≥98% → 양자화 후 97.5% → "0.5%p 하락"이라며 
#      양자화의 실제 영향을 정확히 측정 가능
#
# [핵심] 양자화는 '비교'가 생명입니다.
# - 원본(FP32)이 가장 똑똑한 상태여야만, 양자화(INT8)로 인한 
#   정보 손실이 얼마나 예민하게 발생하는지 정확히 관찰할 수 있습니다.
#
# [실무 가이드]
# 1. 원본은 최선을 다해 학습시킨다: "지능의 손실"만 측정하기 위함입니다.
# 2. 가중치를 안정시킨다: 학습이 덜 되어 날뛰는 가중치는 
#    양자화 눈금(Scale)을 정할 때 큰 오차를 만들어냅니다.
# ----------------------------------------------------------------------
#
# [훈련 파라미터 선택 이유]
# - epochs=1: MNIST는 간단한 데이터셋 → 1에포크로도 98%+ 달성 가능
#   (실제로는 3~5에포크 권장되나, 실습 시간 단축을 위해 1에포크 설정)
# - Adam(lr=0.001): 
#   - 학습률 0.001은 대부분의 CNN에서 안정적인 수렴 보장
#   - 양자화 실험에서는 "동일한 최적화 조건" 유지가 중요 → 비교 공정성 확보
# - batch_size=64: 
#   - GPU 메모리 효율성과 학습 안정성의 균형
#   - 양자화 시에도 동일한 배치 크기 사용 → 배치 정규화 효과 일관성 유지
# ----------------------------------------------------------------------
def train_model(model, loader, epochs=1):
    model.to(device)  # GPU 사용 가능 시 자동으로 이동 (단, 평가는 CPU에서 진행)
    criterion = nn.CrossEntropyLoss()  # MNIST는 다중 클래스 분류 → CrossEntropy
    optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adaptive Moment Estimation
    model.train()  # 훈련 모드 활성화 (Dropout/BatchNorm 동작 변경)
    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()      # 기울기 버퍼 초기화 (중요!)
            outputs = model(images)    # 순전파
            loss = criterion(outputs, labels)  # 손실 계산
            loss.backward()            # 역전파 → 기울기 계산
            optimizer.step()           # 가중치 업데이트
    return model  # 훈련 완료된 모델 반환


# 4. 성능 평가 함수 — "양자화의 진정한 무대: CPU에서의 실행"
# ----------------------------------------------------------------------
# [왜 CPU에서 평가하는가? 양자화의 본질적 목적]
# 💡 양자화의 주 타겟 환경: 
#    - 스마트폰, 임베디드 디바이스, 엣지 디바이스 → 대부분 CPU 기반
#    - "양자화 = CPU 최적화" 라는 공식이 성립
#    - 따라서 평가도 반드시 CPU에서 수행해야 실제 이점 측정 가능
#
# [주의: GPU에서 평가하면 발생하는 문제]
# ❌ "양자화 모델이 오히려 느림"이라는 오해:
#    - GPU는 32비트 연산에 최적화 → 8비트 연산 지원 미비
#    - CPU는 8비트 연산 가속 명령어 (AVX-512 VNNI 등) 보유
#    - → 반드시 CPU에서만 양자화 효과 검증 가능
#
# [평가 프로토콜]
# - model.eval(): Dropout/BatchNorm을 추론 모드로 전환 (중요!)
# - torch.no_grad(): 기울기 계산 비활성화 → 메모리/속도 최적화
# - 배치 단위 처리: 대용량 데이터셋도 메모리 초과 없이 처리
# ----------------------------------------------------------------------
def evaluate_model(model, loader, device='cpu'):
    model.eval()   # 추론 모드 활성화 (학습과 다른 동작)
    model.to(device)  # 반드시 CPU로 이동 (양자화의 실제 타겟 환경)
    correct = 0    # 정답 수 누적
    total = 0      # 전체 샘플 수 누적
    with torch.no_grad():  # 기울기 계산 생략 → 2배 이상 속도 향상
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)  # 모델 추론
            _, predicted = torch.max(outputs.data, 1)  # 가장 높은 확률의 클래스 선택
            total += labels.size(0)                    # 배치 크기 누적
            correct += (predicted == labels).sum().item()  # 정답 수 누적
    return 100 * correct / total  # 정확도 (%) 반환


# 5. 모델 용량 측정 함수 — "압축률을 눈으로 확인하는 유일한 방법"
# ----------------------------------------------------------------------
# [왜 파일 저장 후 크기 측정인가?]
# ❓ "가중치 파라미터 수 × 4바이트 = 용량" 으로 계산하면 안 되는가?
#    → 실제 저장 시 오버헤드(메타데이터, 텐서 구조 정보 등)가 추가됨
#    → 정확한 비교를 위해 "실제 디스크에 저장되는 크기" 측정 필수
#
# [측정 프로토콜의 중요성]
# ✅ state_dict()만 저장: 
#    - 모델 아키텍처 정보 제외 → 순수 가중치 크기만 측정
#    - 양자화는 가중치 압축이 목표 → 아키텍처 정보는 비교 대상 아님
# ✅ 임시 파일 즉시 삭제: 
#    - 디스크 공간 낭비 방지
#    - 여러 번 반복 실험 시 파일 충돌 방지
#
# [기대되는 결과]
# - FP32: 파라미터 1개당 4바이트 → 예: 1,000,000 파라미터 = ~4MB
# - INT8 양자화: 파라미터 1개당 1바이트 + Scale/Zero-point 오버헤드 → ~1.1~1.2MB
#   → 이론적 4배 압축률 달성 여부를 이 함수로 검증
# ----------------------------------------------------------------------
def get_model_size(model, path='temp.pth'):
    # state_dict() = 순수 가중치 텐서만 포함 (아키텍처 정보 제외)
    torch.save(model.state_dict(), path)  # 디스크에 실제 저장 → 오버헤드 포함
    size = os.path.getsize(path)          # OS 레벨에서 실제 파일 크기 측정 (바이트)
    os.remove(path)                       # 측정 후 즉시 삭제 → 흔적 제거
    return size                           # 바이트 단위 크기 반환


# ==========================================================
# 실행부: 비교를 위한 '기준선(Baseline)' 생성 — 양자화 실습의 출발점
# ==========================================================
# [왜 이 단계가 양자화 실습의 50%를 차지하는가?]
# 🔑 "측정되지 않는 것은 개선할 수 없다" (What gets measured gets managed)
#    - 양자화는 정확도/속도/용량의 트레이드오프 최적화 과정
#    - 기준선 없이는 "어느 정도 희생이 허용 가능한가?" 판단 불가
#    - 예: "정확도 0.5%p 희생으로 4배 속도 향상" → 기준선이 있어야 의미 있는 결론
#
# [실습 시 주의사항]
# ⚠️ 반드시 동일한 랜덤 시드 사용: 
#    - 훈련/평가 시 랜덤성 제거 → 재현성 확보
# ⚠️ 동일한 테스트 세트 사용: 
#    - 양자화 전/후 반드시 동일한 test_loader 사용 → 공정한 비교
# ⚠️ 동일한 평가 환경: 
#    - 모두 CPU에서 평가 → 장치 차이로 인한 왜곡 방지
# ────────────────────────────────────────────────────────────
print("🚀 [Step 0] 원본 FP32 모델 준비 및 성적표 산출...")
model_fp32 = SimpleCNN()  # 32비트 부동소수점 기반 원본 모델 생성
model_fp32 = train_model(model_fp32, train_loader)  # 충분한 정확도 확보를 위한 훈련

# ✅ 두 가지 핵심 지표 측정 (양자화 효과 평가의 유일한 기준)
acc_fp32 = evaluate_model(model_fp32, test_loader, device='cpu')  # 정확도 기준선
size_fp32 = get_model_size(model_fp32)                            # 용량 기준선

# 📊 기준선 결과 시각화 — 이 숫자들이 이후 모든 양자화 실험의 '어머니'가 됨
print("-" * 50)
print(f"✅ 원본(FP32) 최종 정확도: {acc_fp32:.2f}%")
print(f"✅ 원본(FP32) 모델 용량: {size_fp32/1024:.2f} KB (≈ {size_fp32/1024/1024:.2f} MB)")
print("-" * 50)
print("💡 준비 완료! 이제 이 탄탄한 베이스라인을 바탕으로 8비트 압축을 시작해봅시다.")
print("   → 다음 단계: [Step 1] 정적 양자화 (Static Quantization) 적용")
# ────────────────────────────────────────────────────────────

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 476kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.17MB/s]


기본 FP32 모델 훈련 중...
FP32 Model Accuracy: 98.34% | Size: 1650.03 KB


## 3. PyTorch Dynamic Quantization
가장 간단한 양자화 방법으로, 주로 Linear(FC) 레이어에 적용합니다.

In [ ]:
print("PyTorch Dynamic Quantization 적용...")

# [핵심 함수] quantize_dynamic: 모델의 가중치를 미리 INT8로 바꾸고, 
# 활성화 값(Activation)은 추론 시점에 '동적'으로 계산합니다.
model_dynamic = torch.quantization.quantize_dynamic(
    # 1. 대상 모델: 현재 최적화의 기준이 되는 FP32 원본 모델을 넣습니다.
    # [참고] 현재 PyTorch의 양자화 백엔드는 CPU에서 가장 안정적이므로 .to('cpu')를 권장합니다.
    model_fp32.to('cpu'),
    
    # 2. 타켓 레이어 {nn.Linear}: 어떤 종류의 레이어를 양자화할지 지정합니다.
    # [중요] Dynamic Quantization은 주로 'Linear(Linear)'와 'RNN' 계열에 가장 효과적입니다.
    # Conv2d 레이어는 지원하지 않거나 효과가 적어 보통 Static Quantization을 씁니다.
    {nn.Linear},  
    
    # 3. 데이터 타입: 가중치를 어떤 형식으로 저장할지 정합니다. (기본 8비트 정수)
    dtype=torch.qint8
)

# 성능 및 크기 측정
acc_dynamic = evaluate_model(model_dynamic, test_loader, device='cpu')
size_dynamic = get_model_size(model_dynamic)

print(f"Dynamic Model Accuracy: {acc_dynamic:.2f}% | Size: {size_dynamic/1024:.2f} KB")

PyTorch Dynamic Quantization 적용...


/tmp/ipython-input-1946685115.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_dynamic = torch.quantization.quantize_dynamic(


Dynamic Model Accuracy: 98.37% | Size: 471.57 KB


## 4. PyTorch Static Quantization
모델의 구조를 수정(QuantStub/DeQuantStub)하고 Calibration 과정을 거쳐 양자화합니다.

- Static Quantization은 실제 추론(inference)에서 사용할 INT8 양자화 모델을 만드는 방식입니다.
- 모델에 **QuantStub / DeQuantStub** 를 추가하여, 입력을 INT8로 바꾸는 구간(Quantize)과 출력에서 다시 FP32로 복원하는 구간(DeQuantize)을 명확히 합니다.
- **Calibration(보정 과정)**: 모델을 변환하기(`convert`) 전에, **실제 데이터(Train Loader)를 모델에 살짝 통과**시킵니다.
    * 이 과정은 학습(Backpropagation)이 아니라, 데이터가 각 레이어를 지날 때 값의 범위(Min-Max)가 어떻게 되는지 관찰하여 양자화 파라미터(Scale, Zero-point)를 확정 짓기 위함입니다.

In [ ]:
import torch
import torch.nn as nn

class QuantizableCNN(nn.Module):
    def __init__(self):
        super(QuantizableCNN, self).__init__()
        # [중요: 양자화 경계 설정] 
        # FP32 세상과 INT8 세상 사이의 '게이트' 역할을 합니다.
        # QuantStub: 입력 데이터(FP32)를 받아서 관찰된 Scale/ZP에 따라 INT8로 변환합니다.
        self.quant = torch.quantization.QuantStub()   
        # DeQuantStub: 연산 결과(INT8)를 다시 다음 프로세스를 위해 FP32로 되돌립니다.
        self.dequant = torch.quantization.DeQuantStub() 
        
        # [Fusion을 위한 구조 설계]
        # F.relu를 쓰지 않고 nn.ReLU를 개별 멤버로 선언하는 이유는 
        # 'Conv+BN+ReLU'를 하나로 합치는(Fusion) 최적화 도구가 모듈 단위로 인식하기 때문입니다.
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU() 
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # 1. 양자화 시작점: 여기서부터 아래 연산들은 모두 INT8 하드웨어 가속을 사용하게 됩니다.
        x = self.quant(x)
        
        # 2. INT8 연산 구간: 이 구역 내의 레이어들은 Calibration을 통해 얻은 Scale로 동작합니다.
        x = self.pool(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool(self.relu2(self.bn2(self.conv2(x))))
        x = x.reshape(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        
        # 3. 양자화 종료점: 최종 결과를 사람이 읽거나 다른 FP32 연산과 결합하기 위해 복원합니다.
        x = self.dequant(x)
        return x

    def fuse_model(self):
        """
        [Operator Fusion: 연산 통폐합 기술]
        ------------------------------------------------------------
        1. 수학적 원리: Conv와 BN은 둘 다 곱셈/덧셈 연산입니다. BN의 계산식을 결합법칙처럼, 
           BN의 복잡한 계산식을 미리 Conv의 가중치(Weight) 안에 계산해서 연산 단계 자체를 줄입니다.
        2. 양자화 시 이점: 부품마다 발생하는 '반올림 오차'를 최소화합니다.
           (오차 누적 지점을 3곳에서 1곳으로 줄여 정확도 유지를 돕습니다.)
        3. 하드웨어 효율: 메모리에서 데이터를 넣었다 뺐다 하는 횟수를 획기적으로 줄여
           추론 속도를 2~3배 이상 끌어올리는 핵심 최적화 과정입니다.
        ------------------------------------------------------------
        """
        # [Conv -> BN -> ReLU] 세 부품을 하나의 '고성능 퓨전 레이어'로 용접합니다.
        torch.quantization.fuse_modules(
            self, 
            [['conv1', 'bn1', 'relu1'], ['conv2', 'bn2', 'relu2']], 
            inplace=True
        )

# ==========================================================
# 🚀 Static Quantization 파이프라인 (5단계)
# ==========================================================

# 1. 훈련 (Training): 먼저 정확도가 높은 FP32 가중치가 준비되어 있어야 합니다.
print("1. 원본 모델(FP32) 준비 및 가중치 훈련 중...")
model_static_base = QuantizableCNN()
model_static_base = train_model(model_static_base, train_loader) # 기존 로직대로 학습

# 2. 평가 모드 전환 및 레이어 합치기 (Fusion)
# static quantization은 실시간 통계를 내지 않으므로 반드시 eval() 모드여야 합니다.
model_static_base.eval()
model_static_base.to('cpu') # 현재 PyTorch CPU 양자화는 CPU 기반 백엔드에서 가장 잘 작동합니다.
model_static_base.fuse_model() # 앞서 정의한 최적화(Fusion) 수행

# 3. QConfig 설정 및 관찰자(Observer) 삽입 (Prepare)
# QConfig는 '어떤 방식으로 숫자를 관찰할지'에 대한 설정입니다.
# fbgemm: 인텔/AMD 등 x86 CPU 환경에서 최적화된 성능을 냅니다.
model_static_base.qconfig = torch.quantization.get_default_qconfig('fbgemm')

# prepare 단계에서는 모델 내부에 'Observer'라는 센서를 매답니다. 
# 이 센서는 앞으로 들어올 데이터의 최댓값/최솟값을 기록합니다.
torch.quantization.prepare(model_static_base, inplace=True)

# 4. Calibration (보정/관찰): Static Quantization의 핵심!
# 실제 데이터(Train set 일부)를 모델에 통과시켜 각 레이어의 Activation 분포를 관찰합니다.
# "이 레이어는 보통 0~5 사이의 값이 나오는군!" 하는 통계를 내서 Scale과 Zero-point를 정합니다.
print("2. Calibration 진행 중 (데이터 분포 관찰)...")
with torch.no_grad():
    for i, (images, _) in enumerate(train_loader):
        if i > 20: break # 보통 10~100 배치 정도면 충분히 정확한 통계가 나옵니다.
        model_static_base(images)

# 5. 최종 변환 (Convert)
# 'Observer' 센서를 제거하고, 실제 FP32 가중치를 INT8 가중치로 완전히 교체합니다.
# 이제 모델 사이즈가 1/4로 줄어들고 CPU에서 훨씬 빠르게 동작하는 최종 양자화 모델이 탄생합니다.
print("3. 최종 모델 변환(Convert to INT8)...")
torch.quantization.convert(model_static_base, inplace=True)

# 결과 확인
acc_static = evaluate_model(model_static_base, test_loader, device='cpu')
size_static = get_model_size(model_static_base)
print(f"📊 최종 결과")
print(f"Static Model Accuracy: {acc_static:.2f}%")
print(f"Weight Size: {size_static/1024:.2f} KB (FP32 대비 약 75% 감소)")

Static Quantization용 모델 훈련 중...


/tmp/ipython-input-698710756.py:40: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(model_static_base, inplace=True)


Calibration 진행 중...


/tmp/ipython-input-698710756.py:50: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.convert(model_static_base, inplace=True)


Static Model Accuracy: 98.55% | Size: 423.49 KB


## 5. ONNX Runtime Quantization: ONNX Runtime 양자화
ONNX로 변환 후 Dynamic Quantization을 수행합니다. `ConvInteger` 에러를 방지하기 위해 `op_types_to_quantize`를 설정했습니다.

In [ ]:
# onnxscript 설치 (최신 onnxruntime 의존성)
!pip install onnxscript -q

In [ ]:
print("\n[Step 4] ONNX Runtime Quantization 진행...")
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType, shape_inference

# 📂 경로 설정: 모델 파일들을 체계적으로 관리합니다.
os.makedirs('models', exist_ok=True)
onnx_path = "models/mnist_fp32.onnx"
onnx_pre_path = "models/mnist_fp32_preprocessed.onnx" # 전처리(Shape 확정)된 모델
onnx_quant_path = "models/mnist_int8_dynamic.onnx"    # 최종 양자화 모델

# ---------------------------------------------------------
# 1. PyTorch -> ONNX Export (브릿지 구축)
# PyTorch 전 전용 모델을 모든 프레임워크가 읽을 수 있는 '공용 포맷(ONNX)'으로 변환합니다.
# ---------------------------------------------------------
dummy_input = torch.randn(1, 1, 28, 28, device='cpu') # 모델 구조 파악을 위한 가짜 입력
torch.onnx.export(
    model_fp32.to('cpu'),
    dummy_input,
    onnx_path,
    input_names=['input'],             # 입력 노드의 이름 (나중에 Inference 시 사용)
    output_names=['output'],           # 출력 노드의 이름
    # dynamic_axes: 배치 사이즈를 고정하지 않고 가변적으로(1, 8, 32 등) 쓸 수 있게 허용함
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=13                   # ONNX의 버전(최신 기능을 쓰기 위해 13 이상 권장)
)
print("1. ONNX Export 완료 (PyTorch -> 공용 포맷 변환)")

# ---------------------------------------------------------
# 2. Shape Inference & Preprocessing (그래프 최적화 준비)
# 양자화 도구가 모델의 각 레이어를 'INT8'로 바꿀 때, 
# 각 데이터가 어떤 모양(Shape)으로 흐르는지 명확히 알아야 최적화가 가능합니다.
# ---------------------------------------------------------
print("2. ONNX Shape Inference(전처리) 수행 중...")
shape_inference.quant_pre_process(
    input_model_path=onnx_path,
    output_model_path=onnx_pre_path,   # 결과를 별도 파일로 저장
    skip_symbolic_shape=False          # 심볼릭 쉐이프(가변 크기)도 추론에 포함
)
print("   - 전처리 완료: 양자화 시 오차를 줄이기 위해 그래프 구조를 정돈함")

# ---------------------------------------------------------
# 3. Dynamic Quantization (실시간 양자화 수행)
# 가중치(Weight)는 미리 INT8로 바꿔두고, 
# 활성화 함수(Activation)는 데이터가 들어올 때 실시간으로 범위를 계산해 양자화합니다.
# ---------------------------------------------------------
print("3. ONNX Dynamic Quantization 수행 중...")
quantize_dynamic(
    model_input=onnx_pre_path,              # 전처리된 모델을 입력으로 사용
    model_output=onnx_quant_path,           # 최종 결과물
    weight_type=QuantType.QInt8,            # 가중치를 8비트 정수로 변환
    # op_types_to_quantize: 양자화했을 때 가장 이득이 큰 연산(Linear, MatMul) 지정
    # 'MatMul'(행렬곱)과 'Gemm'(Linear 레이어)이 모델 무게의 90% 이상을 차지함
    op_types_to_quantize=['MatMul', 'Gemm'] 
)

print(f"✅ ONNX 양자화 완료: {onnx_quant_path}")
print(f"📦 모델 용량: {os.path.getsize(onnx_quant_path)/1024:.2f} KB")

# ---------------------------------------------------------
# 4. ONNX Inference Test (성능 검증)
# PyTorch가 아닌 'ONNX Runtime 엔진'을 사용하여 실제 속도와 정확도를 테스트합니다.
# ---------------------------------------------------------
def onnx_evaluate(onnx_file, loader):
    # 세션 옵션 설정: 하드웨어 최적화를 최대한 활용하도록 설정
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    # InferenceSession: 모델을 메모리에 올리고 실행할 엔진 인스턴스
    # CPUExecutionProvider: CPU 환경에서 실행하겠다는 선언
    session = ort.InferenceSession(onnx_file, sess_options, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name # 'input'이라는 이름을 가져옴
    
    correct = 0
    total = 0

    for images, labels in loader:
        # session.run: 실제 추론 수행 (PyTorch의 model(x)와 같은 역할)
        # 입력을 numpy 배열로 넣어주는 것이 포인트!
        outputs = session.run(None, {input_name: images.numpy()})[0]
        predicted = np.argmax(outputs, axis=1)
        total += labels.size(0)
        correct += (predicted == labels.numpy()).sum()
        
    return 100 * correct / total

acc_onnx = onnx_evaluate(onnx_quant_path, test_loader)
print(f"📊 ONNX Runtime INT8 Accuracy: {acc_onnx:.2f}%")


[Step 4] ONNX Runtime Quantization 진행...


W0122 00:44:36.222000 403 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:68: adapter_lookup: Assertion `false`

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
1. ONNX Export 완료
2. ONNX Shape Inference(전처리) 수행 중...
   - 전처리 완료 (Shape 정보 고정됨)
3. 양자화 수행 중...
ONNX Quantized Model saved at: models/mnist_int8_dynamic.onnx
ONNX Size: 476.70 KB
ONNX Runtime INT8 Accuracy: 98.34%
